# Colab 03 — Ablaciones del modelo original
Quita **un** componente por variante, con la configuración del modelo completo y 5 semillas:
`lacca_eje_fijo` (eje en la línea media), `lacca_sin_asimetria` (D = A), `lacca_global` (atención global).
15 corridas, ~3–4 h. No lee el test. `lacca_global` usa más memoria: si hay *out of memory*, la celda 7 baja el lote a 4 (se declara en el informe).

In [ ]:
# Celda 1 — GPU
import os
!nvidia-smi | head -15

In [ ]:
# Celda 2 — dependencias
%pip install -q segmentation-models-pytorch monai einops timm scipy scikit-image pandas opencv-python-headless tqdm pytest matplotlib onnx onnxruntime
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
assert torch.cuda.is_available(), "Sin GPU: activa una GPU (Runtime → Change runtime type)"

In [ ]:
# Celda 3 — montar Drive y fijar rutas
from google.colab import drive
drive.mount('/content/drive')
CARPETA_DRIVE = "/content/drive/MyDrive/proyecto_tb"      # AJUSTAR si cambia
RAIZ_LOCAL = "/content/proyecto_tb"                        # copia local (rápida)
RES_HPO = f"{CARPETA_DRIVE}/resultados"                    # donde quedó la búsqueda (hpo/)
RES_E3 = f"{CARPETA_DRIVE}/resultados_e3"                  # resultados finales del Entregable 3
HPO_CFG = f"{RES_HPO}/hpo/mejor_config.json"
SPL = f"{RAIZ_LOCAL}/datos/splits"
os.makedirs(RES_E3, exist_ok=True)
assert os.path.isdir(f"{CARPETA_DRIVE}/codigo"), "Falta codigo/ en Drive"
assert os.path.isdir(f"{CARPETA_DRIVE}/datos/splits"), "Falta datos/splits"
print("Drive OK:", CARPETA_DRIVE, "\nResultados E3:", RES_E3)

In [ ]:
# Celda 4 — copiar código y datos a disco local: SOLO train/val (el test no se copia)
import csv, shutil
def a_local(r, raiz):
    n = r.replace("\\", "/"); i = n.find("datos/"); assert i >= 0, r
    return os.path.join(raiz, n[i:])
shutil.copytree(f"{CARPETA_DRIVE}/codigo", f"{RAIZ_LOCAL}/codigo", dirs_exist_ok=True)
os.makedirs(SPL, exist_ok=True)
for nombre in ("train", "val"):
    p = f"{CARPETA_DRIVE}/datos/splits/{nombre}.csv"
    if not os.path.exists(p): print("  (no existe", nombre, ")"); continue
    filas = list(csv.DictReader(open(p, newline="", encoding="utf-8")))
    campos = list(filas[0].keys())
    for f in filas:
        dst, src = a_local(f["ruta"], RAIZ_LOCAL), a_local(f["ruta"], CARPETA_DRIVE)
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(src, dst)
        f["ruta"] = dst
    with open(f"{SPL}/{nombre}.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=campos)
        w.writeheader(); w.writerows(filas)
    print(f"  {nombre:16s} {len(filas)} filas")
for mf in ("split_manifest_v2.json", "split_manifest.json"):
    if os.path.exists(f"{CARPETA_DRIVE}/datos/splits/{mf}"):
        shutil.copy2(f"{CARPETA_DRIVE}/datos/splits/{mf}", f"{SPL}/{mf}")

import sys; sys.path.insert(0, f"{RAIZ_LOCAL}/codigo")

In [ ]:
# Celda — congelar / verificar protocolo.md (SHA-256)
import hashlib
p = f"{CARPETA_DRIVE}/docs/protocolo.md"
assert os.path.exists(p), "Falta docs/protocolo.md en Drive"
h = hashlib.sha256(open(p, "rb").read()).hexdigest()
f = f"{RES_E3}/protocolo.sha256"
if os.path.exists(f):
    assert open(f).read().split()[0] == h, "protocolo.md CAMBIÓ después de congelarse: registra una enmienda"
    print("protocolo verificado:", h[:16], "…")
else:
    open(f, "w").write(h + "  protocolo.md\n"); print("protocolo CONGELADO:", h)

In [ ]:
# Celda — pruebas unitarias del modelo original (control antes de gastar GPU)
!cd "{RAIZ_LOCAL}" && python -m pytest codigo/tests -q 2>&1 | tail -4

In [ ]:
# Celda 5 — configuración: las variantes heredan lr/wd/lote/hiperparámetros del modelo completo
import json
base = json.load(open(HPO_CFG))["lacca_unet"]
abl = {v: dict(base) for v in ["lacca_eje_fijo", "lacca_sin_asimetria", "lacca_global"]}
json.dump(abl, open(f"{RES_E3}/config_ablaciones.json", "w"), indent=2)
print("config base:", {k: base[k] for k in ("lr", "weight_decay", "batch_size", "params_modelo")})

In [ ]:
# Celda 6 — plan
!python "{RAIZ_LOCAL}/codigo/driver_entrenamiento.py" --dry-run --modelos lacca_eje_fijo lacca_sin_asimetria lacca_global --config-hpo "{RES_E3}/config_ablaciones.json" --splits-dir "{SPL}" --resultados "{RES_E3}" | tail -20

In [ ]:
# Celda 7 — ENTRENAR las variantes (repite hasta "Nada pendiente")
!python "{RAIZ_LOCAL}/codigo/driver_entrenamiento.py" --modelos lacca_eje_fijo lacca_sin_asimetria lacca_global --config-hpo "{RES_E3}/config_ablaciones.json" --splits-dir "{SPL}" --resultados "{RES_E3}" --max-minutos 330

In [ ]:
# Celda 8 — SOLO si lacca_global dio out of memory: lote 4 (declarar en el informe)
cfg = json.load(open(f"{RES_E3}/config_ablaciones.json"))
cfg["lacca_global"]["batch_size"] = 4
json.dump(cfg, open(f"{RES_E3}/config_ablaciones_global_b4.json", "w"), indent=2)
!python "{RAIZ_LOCAL}/codigo/driver_entrenamiento.py" --modelos lacca_global --config-hpo "{RES_E3}/config_ablaciones_global_b4.json" --splits-dir "{SPL}" --resultados "{RES_E3}" --max-minutos 330